# ICC World Cup — Win/Loss Summary Per Team

**Question (Aktil Bansal Q-1):**
Given a table `icc_world_cup` with columns `Team_1`, `Team_2`, and `Winner`, write a query (in both SQL and PySpark) that shows each team's total matches played, total wins, and total losses, sorted by wins in descending order.

In [0]:
%sql
-- Step 1: Create the icc_world_cup table in the b_sql.b_practice schema
-- This table stores match results: which two teams played and who won.
create table b_sql.b_practice.icc_world_cup
(
Team_1 Varchar(20),
Team_2 Varchar(20),
Winner Varchar(20)
);
INSERT INTO b_sql.b_practice.icc_world_cup values('India','SL','India');
INSERT INTO b_sql.b_practice.icc_world_cup values('SL','Aus','Aus');
INSERT INTO b_sql.b_practice.icc_world_cup values('SA','Eng','Eng');
INSERT INTO b_sql.b_practice.icc_world_cup values('Eng','NZ','NZ');
INSERT INTO b_sql.b_practice.icc_world_cup values('Aus','India','India');

-- Step 2: Insert sample match records (5 matches across India, SL, Aus, SA, Eng, NZ)
INSERT INTO b_sql.b_practice.icc_world_cup values('India','SL','India');
INSERT INTO b_sql.b_practice.icc_world_cup values('SL','Aus','Aus');
INSERT INTO b_sql.b_practice.icc_world_cup values('SA','Eng','Eng');
INSERT INTO b_sql.b_practice.icc_world_cup values('Eng','NZ','NZ');
INSERT INTO b_sql.b_practice.icc_world_cup values('Aus','India','India');

-- Step 3: Verify the inserted data
select * from b_sql.b_practice.icc_world_cup;

In [0]:
%sql
-- SQL Solution: Calculate per-team win/loss summary using a CTE
-- The CTE 'cte' unpivots the match table: each match contributes two rows
--   (one for Team_1, one for Team_2), each with a win_flag (1 if that team won, 0 otherwise).
-- The outer query groups by team and computes:
--   total_match = count of all appearances
--   win        = sum of win_flag (how many times that team was the winner)
--   loss       = total_match - win
-- Result is ordered by wins descending (best team first).
with cte as(select team_1 as team, case when team_1=winner then 1 else 0 end as win_flag from b_sql.b_practice.icc_world_cup
union all
select team_2, case when team_2=winner then 1 else 0 end as win_flag from b_sql.b_practice.icc_world_cup
order by team)
select team,count(*) as total_match,sum(win_flag) as win, count(*)-sum(win_flag) as loss  from cte group by team order by win desc

In [0]:
# Import common PySpark SQL functions and types needed for DataFrame transformations
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
# Load the icc_world_cup table from Unity Catalog into a Spark DataFrame
# and display the raw match data
df_match=spark.read.table("b_sql.b_practice.icc_world_cup")
df_match.display()

In [0]:
# PySpark Solution: Same win/loss logic as the SQL CTE, implemented with DataFrame API
#
# Step 1: Unpivot each match into two rows — one per team
#   df_team1: extracts Team_1 with a win_flag (1 if Team_1 == Winner, else 0)
#   df_team2: extracts Team_2 with a win_flag (1 if Team_2 == Winner, else 0)
# Step 2: Union the two DataFrames to get every team's appearances
# Step 3: Group by team and aggregate:
#   total_match = count(*)
#   win         = sum(win_flag)
#   loss        = total_match - win
# Step 4: Order by wins descending (best team first)
#
# df_team1=df_match.withColumn("win_flag",when(col("team_1")==col("Winner"),1).otherwise(0)).select(col("team_1").alias("team"),"win_flag")
# df_team2=df_match.withColumn("win_flag",when(col("team_2")==col("Winner"),1).otherwise(0)).select(col("team_2").alias("team"),"win_flag")

df_team1 = df_match.select(
    col("team_1").alias("team"),
    expr("CASE WHEN team_1 = Winner THEN 1 ELSE 0 END as win_flag")
)
df_team2 = df_match.select(
    col("team_2").alias("team"),
    expr("CASE WHEN team_2 = Winner THEN 1 ELSE 0 END").alias("win_flag")
)

# df_team1 = df_match.select(
#     col("team_1").alias("team"),
#     when(col("team_1") == col("Winner"), 1).otherwise(0).alias("win_flag")
# )
# df_team2 = df_match.select(
#     col("team_2").alias("team"),
#     when(col("team_2") == col("Winner"), 1).otherwise(0).alias("win_flag")
# )
df_team=df_team1.unionAll(df_team2)
df_final=df_team.groupBy("team").agg(count("*").alias("total_match"),sum("win_flag").alias("win"),(count("*")-sum("win_flag")).alias("loss")).orderBy(col("win").desc())
df_final.display()